# Retrieval System Engineering

Build and diagnose a versioned, access-aware retrieval pipeline with lexical and dense rankings.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · The archive remembers every version

Sana maintains a policy library where approved versions are added without deleting history. A reviewer has access to extra attachments. An answer is useful only if it cites material that was both effective at the requested date and appropriate for the requester.

The workplace scenario is fictional.


## Ingest and index evidence


### Lecture 01 · Versioned ingestion and chunk provenance

Track source identity, revision, role, and freshness through chunk creation.

#### Ingest a policy so you can explain what happened later

An amended policy states a new effective date. The indexed copy includes the source text but loses which version it came from.

Retain source identity, revision, and effective and access metadata at chunk creation. Test when the old and new versions should enter retrieval. A reviewer needs to be able to reopen exactly the cited evidence.

#### How it works

Use stable IDs for documents and distinct IDs for revisions or passages. Replace previous revisions predictably and filter by effective date and role before passing evidence downstream. Keep original IDs for citation and traceability.

##### Keep the meaning of a document attached to every chunk

Ingestion turns sources into fragments, but each fragment still belongs to a document, a revision, and an eligibility rule. Use stable identifiers with a clear relationship between the source and its chunks. Record when a revision takes effect and what audience can see it. A document that is present is not necessarily relevant to the requested time.

Segment on structure where possible. A table cell can depend on its row and column headers, and an exception can depend on the paragraph before it. Inspect the stored fragments before training or tuning retrieval. An index cannot restore context discarded at ingestion.

Test revision changes and permissions together. Demonstrate that a new version takes over on the effective date and that a historical lookup can still find earlier eligible evidence. Keep deletion and replacement behavior explicit. A stale fragment left in a secondary index can affect results even after the primary record was updated.

#### Worked example

Policy owners publish a new return window. A search result from the old revision is a freshness failure even if its wording matches the query exactly.

#### Try it

Inspect the bundled versioned policy records. Add a new revision and verify which one is visible to each role.


In [ ]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class Passage:
    document: str
    revision: int
    effective: int
    audience: str
    text: str
    chunk: int = 1

    @property
    def source(self):
        prefix = f"{self.document}@{self.revision}"
        return prefix if self.chunk == 1 else f"{prefix}#{self.chunk}"


corpus = (
    Passage("leave", 1, 1, "public", "vacation leave limit 10 days"),
    Passage("leave", 2, 3, "public", "vacation leave limit 15 days"),
    Passage("leave", 3, 8, "public", "vacation leave limit 18 days"),
    Passage("travel", 1, 1, "public", "travel policy for boarding and rail. keep the receipt"),
    Passage("equipment", 1, 1, "public", "board replacement and circuit equipment"),
    Passage("payroll", 1, 1, "finance", "payroll adjustments and tax forms"),
)


def visible(records, role, as_of):
    latest = {}
    for row in records:
        if row.effective <= as_of and (row.document not in latest or row.revision > latest[row.document].revision):
            latest[row.document] = row
    return [
        replace(row, text=sentence.strip(), chunk=chunk)
        for row in latest.values() if row.audience in {"public", role}
        for chunk, sentence in enumerate(row.text.split("."), 1) if sentence.strip()
    ]


eligible = visible(corpus, "employee", 6)
assert {row.source for row in eligible} == {"leave@2", "travel@1", "travel@1#2", "equipment@1"}


**Compare your result**

Return only the latest effective revision and respect access controls. Preserve the document ID in the result.


### Lecture 02 · Lexical retrieval and dense similarity

Compare keyword ranking with a small dense semantic representation.

#### When synonyms help and identifiers hurt

A user alternates between ‘holiday’ and ‘annual leave’ while another asks for a form identified only by a code.

Use the same query labels for lexical and dense methods. Semantic closeness can help synonyms and still fail exact references. Report those slices separately before choosing a hybrid.

#### How it works

Lexical matching rewards shared terms and can miss synonyms. Dense vectors represent related meanings as nearby points. The lab uses manually constructed low-dimensional concept vectors so the rankings can be diagnosed by inspection.

##### Learn why each retrieval signal works

Lexical ranking uses visible terms. Exact matches are particularly important for names, dates, codes, and form numbers. A dense representation can place differently worded descriptions near one another. That helps with synonyms and concepts while sometimes weakening the distinction between related but different identifiers.

Evaluate both signals on the same labeled questions. Inspect one exact reference and one paraphrase before averaging. Make sure candidates are eligible for the user and query date before scoring relevance. A perfect semantic match from a restricted source should never become an ordinary candidate.

The demo uses a small corpus so you can inspect what each representation includes. Note which fields are encoded and whether a title or body changes the result. Transfer those questions to a real corpus before choosing an index or model. Better retrieval comes from a clearer definition of relevance as well as better ranking.

#### Worked example

The query “holidays” should retrieve a policy titled “vacation leave” even though it shares no exact terms.

#### Try it

Score the same labeled cases with both ranking methods and inspect the first result.


In [ ]:
from math import sqrt
import re


def tokens(text):
    return set(re.findall(r"[a-z]+", text.lower()))


concepts = (
    {"vacation", "leave", "holidays"},
    {"travel", "boarding", "rail", "journey", "board"},
    {"board", "equipment", "circuit"},
    {"payroll", "tax", "adjustments"},
)


def vector(text):
    terms = tokens(text)
    return tuple(len(terms & concept) for concept in concepts)


def similarity(left, right):
    norm = sqrt(sum(v * v for v in left) * sum(v * v for v in right))
    return sum(a * b for a, b in zip(left, right)) / norm if norm else 0


def ranked(query, rows, score):
    scores = [(score(query, row.text), row.source) for row in rows]
    return [source for value, source in sorted(scores, key=lambda pair: (-pair[0], pair[1])) if value > 0]


def lexical(query, rows):
    return ranked(query, rows, lambda a, b: len(tokens(a) & tokens(b)))


def dense(query, rows):
    return ranked(query, rows, lambda a, b: similarity(vector(a), vector(b)))


assert lexical("holidays", eligible) == []
assert dense("holidays", eligible)[0] == "leave@2"


**Compare your result**

Recall can improve on vocabulary shifts. Examine a query where the semantic representation is too broad or incomplete.


## Fuse, rank and verify


### Lecture 03 · Hybrid fusion and relevance diagnostics

Fuse two rankings, rerank the candidates, and measure retrieval by stage.

#### Find the lost document or the buried document

Two rankings contain the right record, but after combining them the passage appears below the cutoff. The metric reports a miss either way.

Inspect candidate recall and rank at each stage. Reranking can improve ordering when a passage is present, but it cannot recover a passage removed before scoring. Keep a labeled example of each failure.

#### How it works

Rank fusion combines evidence from different scorers. A second pass can use higher-cost features to reorder a short list. Measure recall at K and reciprocal rank on open, restricted, and no-answer queries, distinguishing answerable queries from safe abstention.

##### Diagnose candidates before tuning the final order

Combine lexical and dense rankings when their strengths complement each other. Rank-based fusion can favor candidates that appear high in either list without requiring compatible score scales. A later reranker can use more context to order the smaller set, at the cost of another processing step.

For a labeled query, inspect the candidate pool before reranking. If the correct passage never arrived, changing the final scoring cannot recover it. If it appears but falls after the cutoff, ranking is a plausible fix. Track recall at each candidate limit and the rank of the first relevant result to separate the two questions.

A practical evaluation includes exact codes, synonyms, restricted sources, and queries with no answer. Compare latency along with relevance. A marginal improvement in order might not justify an additional stage for the routine workload. Keep a few cases that visibly improve and regress so the choice can be explained.

#### Worked example

An exact term match for ‘board’ can favor an equipment policy over a travel policy. An additional concept match can reorder the fused candidates.

#### Try it

Run the retrieval report. Change the fusion parameters and compare results for the synonym and exact-term cases.


In [ ]:
from collections import defaultdict


def fused_scores(query, rows, k):
    scores = defaultdict(float)
    for ranking in (lexical(query, rows), dense(query, rows)):
        for position, source in enumerate(ranking, 1):
            scores[source] += 1 / (k + position)
    return scores


def fusion(query, rows):
    scores = fused_scores(query, rows, k=20)
    return sorted(scores, key=lambda source: (-scores[source], source))


def hybrid(query, rows, k=20, rerank_weight=0.30):
    scores = fused_scores(query, rows, k)
    by_source = {row.source: row for row in rows}
    return sorted(scores, key=lambda source: (
        -(scores[source] + rerank_weight * similarity(vector(query), vector(by_source[source].text))), source
    ))


cases = (("holidays", "employee", "leave@2"), ("board journey", "employee", "travel@1"),
         ("circuit", "employee", "equipment@1"), ("payroll", "finance", "payroll@1"),
         ("payroll", "employee", None), ("astronomy", "employee", None))


def evaluate(method):
    positions, clean = [], 0
    for query, role, wanted in cases:
        ranking = method(query, visible(corpus, role, 6))
        if wanted is None:
            clean += not ranking
        else:
            positions.append(ranking.index(wanted) + 1 if wanted in ranking else None)
    return {"recall@2": sum(pos is not None and pos <= 2 for pos in positions) / len(positions),
            "MRR": sum(1 / pos for pos in positions if pos) / len(positions), "no_evidence": clean}


for method in (lexical, dense, fusion, hybrid):
    print(method.__name__, evaluate(method))
assert evaluate(hybrid)["recall@2"] > evaluate(lexical)["recall@2"]
assert evaluate(hybrid)["MRR"] > evaluate(fusion)["MRR"]
assert evaluate(hybrid)["no_evidence"] == 2


**Compare your result**

Use stage metrics to see whether the relevant document is missing from the candidates or ranked too low.


### Lecture 04 · Citation verification and freshness checks

Allow citations only to eligible retrieved evidence and abstain when evidence is incomplete.

#### Check eligibility again when the citation is shown

A cached answer correctly quotes a passage that is now expired. The current index excludes it, but the answer still displays an old source ID.

Bind cached responses to relevant policy, audience, and revision state or revalidate before reuse. Test stale and denied citations along with a supported quote. An honest abstention is preferable when authoritative evidence is missing.

#### How it works

Validate source IDs against the passages the response received and verify the supporting text. Treat a stale revision as unavailable even if it was present in a prior cached result. Reevaluate after indexes or access policies change.

##### Verify the citation at the boundary where it is used

A citation should point to an eligible source version and support the exact claim. Check that the identifier belongs to the passages made available to the model. Then inspect whether the relevant passage supports the stated amount, date, and exception. The same document title can refer to different content over time.

Update tests when evidence changes. If a result was cached while a source was current, it can outlast the source's permissions or effective date. Bind the cache to scope and revision, or revalidate before returning the answer. Give missing and contradictory evidence distinct outcomes callers can understand.

Try one answer with a correct amount and an incorrect source. Try another with a correct source but an unsupported exception. A useful validator catches both. When evidence remains incomplete, return an explicit handoff. That fallback preserves trust better than presenting a citation that a reviewer cannot reproduce.

#### Worked example

A candidate quotes an old leave policy accurately. The citation is still invalid when that revision is no longer eligible.

#### Try it

Run the freshness and citation fixtures. Add a case where a restricted source has the best lexical match.


In [ ]:
def verify_quote(source, quote, retrieved_sources, eligible_rows):
    evidence = {row.source: row.text for row in eligible_rows if row.source in retrieved_sources}
    if source in evidence and quote.strip() and quote in evidence[source]:
        return quote, source
    return None


hits = hybrid("holidays", eligible)
assert verify_quote("leave@2", "15 days", hits, eligible) == ("15 days", "leave@2")
assert verify_quote("leave@1", "10 days", hits, eligible) is None
assert verify_quote("leave@2", "20 days", hits, eligible) is None
assert verify_quote("payroll@1", "tax forms", hits + ["payroll@1"], eligible) is None
assert verify_quote("leave@2", "15 days", hits, visible(corpus, "employee", 8)) is None


**Compare your result**

The restricted source is filtered before both rankings, and the grounded-answer stage uses only retrieved, currently eligible evidence.


## Course project

Deliver a policy-search pipeline with versioned ingestion, separate lexical and dense retrieval, reranking, and a cited response or abstention.

On labeled queries, report recall and first-relevant-rank results for three configurations. Detect a stale revision, a restricted passage, and a proposed citation outside the retrieved set.
